# 第 9 章实验：卷积核、stride 与边界
在合成图上观察手工卷积核响应。修改参数格后从头运行，不需要任何图片文件。

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

stride = 1
padding = 1
kernel_name = 'vertical'  # 可改为 horizontal
torch.manual_seed(9)

In [ ]:
image = torch.zeros(1, 1, 32, 32)
image[:, :, 7:25, 9:23] = 1.0
image[:, :, torch.arange(5, 28), torch.arange(3, 26)] = 0.5
vertical = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
horizontal = vertical.T.contiguous()
kernel = {'vertical': vertical, 'horizontal': horizontal}[kernel_name].reshape(1, 1, 3, 3)
response = F.conv2d(image, kernel, stride=stride, padding=padding)
formula = (32 + 2 * padding - (3 - 1) - 1) // stride + 1
print('input NCHW:', tuple(image.shape), 'output:', tuple(response.shape), 'formula H/W:', formula)
assert response.shape[-2:] == (formula, formula)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(image[0, 0], cmap='gray', vmin=0, vmax=1); axes[0].set_title('synthetic input')
axes[1].imshow(kernel[0, 0], cmap='coolwarm'); axes[1].set_title(kernel_name + ' kernel')
axes[2].imshow(response[0, 0].detach(), cmap='coolwarm'); axes[2].set_title(f'response {tuple(response.shape[-2:])}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 破坏实验：padding 改变边界假设
比较零填充与反射填充。反射填充不会在边缘突然接上黑色，但也不一定适合所有任务。

In [ ]:
zero_padded = F.conv2d(image, kernel, padding=1)
reflected = F.conv2d(F.pad(image, (1, 1, 1, 1), mode='reflect'), kernel)
edge_difference = (zero_padded - reflected).abs()
print('边界差异均值:', float(edge_difference[..., [0, -1], :].mean()))
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(zero_padded[0, 0], cmap='coolwarm'); axes[0].set_title('zero padding')
axes[1].imshow(reflected[0, 0], cmap='coolwarm'); axes[1].set_title('reflection padding')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
layers = [(3, 1), (2, 2), (3, 1)]
r, jump = 1, 1
for kernel_size, layer_stride in layers:
    r = r + (kernel_size - 1) * jump
    jump *= layer_stride
    print(f'kernel={kernel_size}, stride={layer_stride} -> receptive_field={r}, jump={jump}')